# ForgeGraph AI - Dataset exploration

This is the first notebook for ForgeGraph. The goal is to understand the structure of the MaintIE dataset before trying any model training.

**Dataset:** [MaintIE](https://github.com/nlp-tlp/maintie) (Bikaun et al., 2024)  
**Paper:** [LREC-COLING 2024](https://aclanthology.org/2024.lrec-main.954/)  
**File:** `data/gold_release.json` from the authors' repository.

We'll inspect records, entity labels, and relationships. This notebook doesn't train a model.

## 1. Load the data

Each record is a short maintenance note with tokens, entity annotations, and relations. We can load it directly from the original public GitHub repository. No libraries need to be installed for this notebook.

In [1]:
import json
from collections import Counter
from urllib.request import urlopen

DATA_URL = "https://raw.githubusercontent.com/nlp-tlp/maintie/main/data/gold_release.json"

with urlopen(DATA_URL, timeout=30) as response:
    records = json.load(response)

print("Number of records:", len(records))
print("Fields in first record:", list(records[0].keys()))

Number of records: 1076
Fields in first record: ['text', 'tokens', 'entities', 'relations']


## 2. Look at an actual maintenance record

We'll print one record and its annotations. In MaintIE, entity `start` is included and `end` is excluded, so `tokens[start:end]` gives the annotated phrase.

In [2]:
record = records[0]

print("Text:", record["text"])
print("Tokens:", record["tokens"])

print("\nEntities:")
for entity in record["entities"]:
    phrase = " ".join(record["tokens"][entity["start"]:entity["end"]])
    print(f"  {phrase!r} -> {entity['type']}")

print("\nRelations:")
for relation in record["relations"]:
    source = record["entities"][relation["head"]]
    target = record["entities"][relation["tail"]]
    source_text = " ".join(record["tokens"][source["start"]:source["end"]])
    target_text = " ".join(record["tokens"][target["start"]:target["end"]])
    print(f"  {source_text!r} --{relation['type']}--> {target_text!r}")

Text: <id> air conditioner thermostat not working
Tokens: ['<id>', 'air', 'conditioner', 'thermostat', 'not', 'working']

Entities:
  'air conditioner' -> PhysicalObject/EmittingObject/ElectricCoolingObject
  'thermostat' -> PhysicalObject/SensingObject/TemperatureSensingObject
  'not working' -> State/UndesirableState/FailedState

Relations:
  'air conditioner' --hasPart--> 'thermostat'
  'not working' --hasParticipant/hasPatient--> 'thermostat'


## 3. Count entity categories

MaintIE has detailed labels such as `PhysicalObject/SensingObject/TemperatureSensingObject`. For a simple overview, we'll count only the first part of each label, like `PhysicalObject`.

In [3]:
entity_counts = Counter()
for record in records:
    for entity in record["entities"]:
        top_level_label = entity["type"].split("/")[0]
        entity_counts[top_level_label] += 1

print("Top-level entity counts:")
for label, count in entity_counts.most_common():
    print(f"  {label}: {count}")

Top-level entity counts:
  PhysicalObject: 1994
  Activity: 784
  State: 438
  Process: 146
  Property: 35


## 4. Count relation types

Relations describe connections between pairs of entities. Let's see which types occur most often.

In [4]:
relation_counts = Counter()
for record in records:
    for relation in record["relations"]:
        relation_counts[relation["type"]] += 1

print("Relation counts:")
for label, count in relation_counts.most_common():
    print(f"  {label}: {count}")

Relation counts:
  hasParticipant/hasPatient: 1206
  hasPart: 533
  isA: 364
  hasParticipant/hasAgent: 166
  contains: 38
  hasProperty: 34


## 5. Check the annotation indices

Before model training, it is useful to check whether entity spans and relation indices point to valid entries. This is a basic data-quality check, not a full validation of the annotations.

In [5]:
invalid_spans = 0
invalid_relations = 0

for record in records:
    tokens = record["tokens"]
    entities = record["entities"]

    for entity in entities:
        if not (0 <= entity["start"] < entity["end"] <= len(tokens)):
            invalid_spans += 1

    for relation in record["relations"]:
        if not (0 <= relation["head"] < len(entities)
                and 0 <= relation["tail"] < len(entities)):
            invalid_relations += 1

print("Invalid entity spans:", invalid_spans)
print("Invalid relation indices:", invalid_relations)

Invalid entity spans: 0
Invalid relation indices: 0


## My observations

- Number of records loaded: 1,076
- Most common entity type: PhysicalObject (1,994)
- Most common relation type: hasParticipant/hasPatient (1,206)
- Interesting example: "air conditioner thermostat not working" shows how an equipment component and its condition are connected.
- Something I want to understand better: How can a model learn to identify entities and their relationships in maintenance records it has never seen before?

In [6]:

# Step 6: Inspect a few more maintenance records

for i, record in enumerate(records[1:5], start=2):
    print(f"\nRecord {i}: {record['text']}")

    for entity in record["entities"]:
        phrase = " ".join(
            record["tokens"][entity["start"]:entity["end"]]
        )
        category = entity["type"].split("/")[0]
        print(f"  {phrase} -> {category}")

    print("-" * 45)



Record 2: <id> air conditioner thermostat unserviceable
  air conditioner -> PhysicalObject
  thermostat -> PhysicalObject
  unserviceable -> State
---------------------------------------------

Record 3: <id> air conditioner unserviceable
  air conditioner -> PhysicalObject
  unserviceable -> State
---------------------------------------------

Record 4: <id> air conditioner unserviceable when stationary
  air conditioner -> PhysicalObject
  unserviceable -> State
---------------------------------------------

Record 5: <id> air conditioner warm
  air conditioner -> PhysicalObject
  warm -> State
---------------------------------------------


In [7]:

# Step 7: Explore random maintenance records

import random

random.seed(42)

sample_records = random.sample(records, 5)

for i, record in enumerate(sample_records, start=1):
    print(f"\nExample {i}: {record['text']}")

    for entity in record["entities"]:
        phrase = " ".join(
            record["tokens"][entity["start"]:entity["end"]]
        )

        category = entity["type"].split("/")[0]

        print(f"  {phrase} -> {category}")

    print("-" * 45)



Example 1: <id> cracks in back right hand side handrail
  cracks -> Property
  handrail -> PhysicalObject
---------------------------------------------

Example 2: <id> beacon unserviceable
  beacon -> PhysicalObject
  unserviceable -> State
---------------------------------------------

Example 3: <id> - change out jockey pulley bearing
  change out -> Activity
  jockey pulley -> PhysicalObject
  pulley -> PhysicalObject
  bearing -> PhysicalObject
---------------------------------------------

Example 4: <id> - CW right hand hoist cylinder leaking
  leaking -> Process
  hoist cylinder -> PhysicalObject
---------------------------------------------

Example 5: <id> replace timing sensor
  replace -> Activity
  timing sensor -> PhysicalObject
---------------------------------------------


In [8]:

# Step 8: Check how often entities overlap

overlap_records = []
overlap_pairs = 0

for record in records:
    entities = record["entities"]
    pairs = []

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):
            a = entities[i]
            b = entities[j]

            if max(a["start"], b["start"]) < min(a["end"], b["end"]):
                pairs.append((i, j))

    if pairs:
        overlap_records.append((record, pairs))
        overlap_pairs += len(pairs)

print("Total records:", len(records))
print("Records with overlapping entities:", len(overlap_records))
print("Overlapping entity pairs:", overlap_pairs)
print("Percentage of affected records:",
      round(100 * len(overlap_records) / len(records), 2), "%")

print("\nSome examples:")

for record, pairs in overlap_records[:3]:
    print("\nText:", record["text"])

    for i, j in pairs:
        a = record["entities"][i]
        b = record["entities"][j]

        phrase_a = " ".join(record["tokens"][a["start"]:a["end"]])
        phrase_b = " ".join(record["tokens"][b["start"]:b["end"]])

        print(f"  '{phrase_a}' overlaps with '{phrase_b}'")


Total records: 1076
Records with overlapping entities: 328
Overlapping entity pairs: 385
Percentage of affected records: 30.48 %

Some examples:

Text: <id> air filters plugged
  'air filters' overlaps with 'filters'

Text: <id> air horn disconnected
  'air horn' overlaps with 'horn'

Text: <id> air horn doesn't work
  'air horn' overlaps with 'horn'



## Findings from dataset exploration

- Loaded 1,076 maintenance records from MaintIE.
- Found five main entity categories.
- PhysicalObject is the most common entity category.
- Found 2,341 annotated relationships.
- All checked entity spans and relation indices were valid.
- Found 328 records containing overlapping entities (30.48%).
- Some records contain nested entities, such as "air filters" and "filters".

### Next steps

I plan to explore span-based entity recognition because overlapping entities cannot be fully represented using standard single-layer BIO tagging.

Before training, I also need to prepare the data splits and check for similar or repeated maintenance records that could affect evaluation.
